### **Notebook 1 — Data Cleaning & Preprocessing (Python)**

 Purpose :
This notebook performs all cleaning and preprocessing for the NorthStar dataset.


Objectives:
- Load all raw CSV datasets from the GitHub repository  
- Assess data quality  
- Clean missing, duplicated, inconsistent values  
- Validate key relationships  
- Perform feature engineering where necessary  
- Export cleaned datasets for use in:
  - Notebook 2 (SQL in R + R Analytics)
  - Notebook 3 (Python Analytics)
  - Notebook 4 (MongoDB Atlas)


In [ ]:
# Install dependencies if needed
!pip install pandas numpy matplotlib seaborn

**Interpretation :**

These libraries support data loading (`pandas`), numerical cleaning (`numpy`), and quick visualization checks (`matplotlib/seaborn`).
They must be installed before running any cleaning code.

 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

**Interpretation :**

These imports form the core cleaning toolkit



*   pandas → dataframes

*   numpy → numeric operations
*   matplotlib/seaborn → light visual profiling if needed


2. Load Raw Datasets from GitHub

The raw datasets are stored inside the GitHub folder: `raw_data/`.

In [ ]:
# GitHub configuration
username = "aqibashraf129-wq"
repo = "northstar-analytics"
branch = "main"

base_url = f"https://raw.githubusercontent.com/{username}/{repo}/{branch}/raw_data/"

files = {
    "customers": "customers.csv",
    "drivers": "drivers.csv",
    "deliveries": "deliveries.csv",
    "vehicles": "vehicles.csv",
    "hubs": "hubs.csv",
    "complaints": "complaints.csv",
    "incidents": "incidents.csv",
    "orders": "orders.csv",
    "app_events": "app_events.csv"
}

datasets = {}

for name, filename in files.items():
    url = base_url + filename
    try:
        datasets[name] = pd.read_csv(url)
        print(f" Loaded {filename} successfully!")
    except Exception as e:
        print(f" Error loading {filename}: {e}")

 Loaded customers.csv successfully!
 Loaded drivers.csv successfully!
 Loaded deliveries.csv successfully!
 Loaded vehicles.csv successfully!
 Loaded hubs.csv successfully!
 Loaded complaints.csv successfully!
 Loaded incidents.csv successfully!
 Loaded orders.csv successfully!
 Loaded app_events.csv successfully!


**Interpretation:**

The datasets were successfully loaded from your GitHub repository using the correct raw URLs. This confirms that the repository structure is correct and ready for further processing.

3. Inspect Dataset Structure

In [ ]:
for name, df in datasets.items():
    print("\n==============================")
    print(f"Dataset: {name.upper()}")
    print(df.head())
    print(df.info())


Dataset: CUSTOMERS
  customer_id  age  home_zone customer_type          signup_date  \
0       C0001   26      North           SME  2024-11-27 04:25:00   
1       C0002   61    AIRPORT      Consumer  2025-10-28 01:04:00   
2       C0003   66       East      Consumer  2025-07-02 03:23:00   
3       C0004   75    CENTRAL      Consumer  2025-08-19 01:58:00   
4       C0005   26  Riverside      Consumer  2025-06-03 06:02:00   

   loyalty_score  app_engagement_score preferred_channel account_status  
0           44.9                  69.2               App         Active  
1           55.4                  66.6               App         Active  
2           75.9                  33.8               NaN         Active  
3           32.5                  33.0               App         Active  
4           55.9                 100.0               Web         Active  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 9 columns):
 #   Column            

**Interpretation:**

This output reveals:
Whether key fields (IDs, timestamps, links) exist
Data types (object, int, float, datetime)
Missing values and inconsistencies
Whether the dataset needs standardisation

This step helps identify cleaning requirements.

4. Cleaning Functions (Reusable)

In [ ]:
def standardise_column_names(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df

def fix_dates(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

def remove_duplicates(df):
    return df.drop_duplicates()

def clean_strings(df):
    str_cols = df.select_dtypes(include=["object"]).columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()
    return df

**Interpretation:**

These reusable cleaning functions enforce consistent formatting across all datasets, reducing errors in downstream SQL, Python, and MongoDB workflows.

5. Apply Cleaning to All Datasets

In [ ]:
cleaned = {}

for name, df in datasets.items():
    df = standardise_column_names(df)
    df = clean_strings(df)
    df = remove_duplicates(df)

    # Attempt automatic date conversion
    date_cols = [col for col in df.columns if "date" in col or "time" in col]
    df = fix_dates(df, date_cols)

    cleaned[name] = df

print("Cleaning complete.")

Cleaning complete.


**Interpretation:**

The cleaning pipeline standardises column names, removes duplicates, trims whitespace, and converts date-related fields into proper datetime format. This ensures uniformity before performing analytical workflows in R, Python, and MongoDB.

6. Check Missing Values

In [ ]:
for name, df in cleaned.items():
    print("\n==============================")
    print(f" Missing Value Summary: {name.upper()}")
    print(df.isnull().sum())


 Missing Value Summary: CUSTOMERS
customer_id              0
age                      0
home_zone                0
customer_type            0
signup_date              0
loyalty_score           20
app_engagement_score     0
preferred_channel        0
account_status           0
dtype: int64

 Missing Value Summary: DRIVERS
driver_id           0
base_zone           0
employment_type     0
years_experience    0
training_score      7
driver_rating       0
shift_preference    0
active_flag         0
dtype: int64

 Missing Value Summary: DELIVERIES
delivery_id                       0
order_id                          0
driver_id                         0
vehicle_id                        0
hub_id                            0
dispatch_time                     0
delivery_completed_at             0
delivery_status                   0
route_distance_km                 0
manual_route_override_count       0
proof_of_completion_missing       0
customer_rating_post_delivery    14
fuel_or_charge_cost

**Interpretation:**

This summary reveals:
Which fields require imputation
Where relationships may be broken (e.g., missing customer_id)
Whether data consistency issues exist

7. Example Fixes

In [ ]:
# Example: Fill missing driver IDs with -1 (if required)
if "driver_id" in cleaned["deliveries"].columns:
    cleaned["deliveries"]["driver_id"] = cleaned["deliveries"]["driver_id"].fillna(-1)

Applying this step allows me to identify incomplete or inconsistent values early, ensuring I understand where relational links may break during later analysis.

8. Export Cleaned Data to CSV (for GitHub Upload)

In [ ]:
output_path = "/content/cleaned/"
import os
os.makedirs(output_path, exist_ok=True)

for name, df in cleaned.items():
    df.to_csv(output_path + f"{name}_cleaned.csv", index=False)

print("All cleaned datasets exported.")

All cleaned datasets exported.
